<a href="https://colab.research.google.com/github/ihemanthc/grievance_triage/blob/main/grievance_triage_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Constituency War Room — Grievance Triage Challenge
### Single notebook: TF-IDF baseline (category + urgency) + optional MuRIL transformer for urgency

**Run this in Google Colab.** Before running: `Runtime → Change runtime type → GPU`
(the TF-IDF section works fine on CPU too, but you'll want GPU for the transformer section later in this notebook).

**Strategy, and why:**
- A chi-square test on `train.csv` shows category and urgency are statistically
  independent (p ≈ 0.12) → train them as **two separate classifiers** (8-way
  category, 3-way urgency) instead of one sparse 24-way head. Each gets far
  more effective signal per class this way.
- **Category is basically solved** by TF-IDF + LinearSVC (~1.00 CV accuracy,
  verified not leakage — no duplicate rows, and a subject-only baseline only
  gets 0.73, so the body text is doing real work).
- **Urgency is the hard part** (~0.94 CV accuracy with TF-IDF). Words like
  "urgent"/"immediate"/"top priority" appear at ~51–53% across *all three*
  true urgency levels — i.e. no signal — because nearly every message claims
  urgency. Urgency has to come from actual content severity.
- The dataset also plants distractor sentences ("our ration shop works fine,
  this is a different matter") that negate other topics. A bag-of-words model
  can't distinguish "mentions X" from "explicitly negates X" — that's the
  hypothesis the transformer section tests, and it's why GPU effort should go
  **only** into urgency, not category.

**Structure of this notebook:**
1. Upload data
2. TF-IDF baseline → produces `submission_baseline_tfidf.csv`
3. (Optional, GPU) Fine-tune MuRIL for urgency only → produces `submission_transformer.csv`
4. Combine / ensemble

## 1. Upload data
Run this cell, then select `train.csv`, `test.csv`, and `sample_submission.csv`
together in the file picker (from the competition's **Data** tab on Kaggle).

In [1]:
from google.colab import files
import os

if not all(os.path.exists(f) for f in ['train.csv', 'test.csv', 'sample_submission.csv']):
    uploaded = files.upload()  # select train.csv, test.csv, sample_submission.csv

assert os.path.exists('train.csv') and os.path.exists('test.csv'), \
    "train.csv / test.csv not found — re-run this cell and upload them."
print("Data files ready.")

Saving train.csv to train.csv
Saving sample_submission.csv to sample_submission.csv
Saving test.csv to test.csv
Data files ready.


In [2]:
import pandas as pd, numpy as np

train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

train[['category', 'urgency']] = train['label'].str.split('|', expand=True)
train['text'] = (train['subject'] + ' . ' + train['body']).str.strip()
test['text']  = (test['subject']  + ' . ' + test['body']).str.strip()

print(train.shape, test.shape)
train.head()

(4800, 10) (2000, 7)


,id,subject,body,channel,district,complaint_history,label,category,urgency,text
0,G102149,Ambulance response time complaint,"Sir namaskara, The aspatre ward toilets are in...",whatsapp,Malligere,repeat_complaint,healthcare|routine,healthcare,routine,Ambulance response time complaint . Sir namask...
1,G103955,Sand lorries running at night,"Dear team, There have been five chain-snatchin...",whatsapp,Krishnapura,escalated,law_and_order|high,law_and_order,high,"Sand lorries running at night . Dear team, The..."
2,G106063,Urgent problem in our area,"Pranam sir, The main rasta through ward 4 in K...",janata_darshan,Ambalpet,first_time,roads_transport|high,roads_transport,high,"Urgent problem in our area . Pranam sir, The m..."
3,G102374,Power cuts of 6+ hours daily in Chandpur,"Sir ji, This month's bills are three to four t...",field_visit,Ambalpet,escalated,electricity|routine,electricity,routine,Power cuts of 6+ hours daily in Chandpur . Sir...
4,G105824,No drinking water in ward 4 for three days now,"Namaskara sir, The new apartment construction ...",field_visit,Devgiri,repeat_complaint,water_supply|routine,water_supply,routine,No drinking water in ward 4 for three days now...


## 2. TF-IDF baseline
Features: word TF-IDF (1–2 grams) + char TF-IDF (3–5 grams, robust to
code-mixed spelling variants) + `complaint_history` one-hot.
`channel` and `district` are excluded — both are near-uniform across labels
(no signal); `complaint_history` does carry signal (first-time complaints
skew routine, repeat complaints skew high).

In [3]:
!pip install -q scikit-learn scipy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from scipy.sparse import hstack, csr_matrix

def build_features(tr_text, va_text, tr_hist, va_hist, word_vec, char_vec):
    Xw_tr = word_vec.fit_transform(tr_text); Xw_va = word_vec.transform(va_text)
    Xc_tr = char_vec.fit_transform(tr_text); Xc_va = char_vec.transform(va_text)
    ht = pd.get_dummies(tr_hist)
    hv = pd.get_dummies(va_hist).reindex(columns=ht.columns, fill_value=0)
    Xtr = hstack([Xw_tr, Xc_tr, csr_matrix(ht.values.astype(float))]).tocsr()
    Xva = hstack([Xw_va, Xc_va, csr_matrix(hv.values.astype(float))]).tocsr()
    return Xtr, Xva

# ---- 5-fold CV check (skip straight to the "fit on full data" cell if you just want to submit fast) ----
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
combo_accs = []
for tr_idx, va_idx in skf.split(train, train['label']):
    tr, va = train.iloc[tr_idx], train.iloc[va_idx]
    wv = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    cv_ = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=3, sublinear_tf=True)
    Xtr, Xva = build_features(tr['text'], va['text'], tr['complaint_history'], va['complaint_history'], wv, cv_)

    cat_m = LinearSVC(class_weight='balanced', C=1.0).fit(Xtr, tr['category'])
    urg_m = LinearSVC(class_weight='balanced', C=1.0).fit(Xtr, tr['urgency'])
    combo_pred = pd.Series(cat_m.predict(Xva)).astype(str) + '|' + pd.Series(urg_m.predict(Xva)).astype(str)
    combo_accs.append(accuracy_score(va['label'].values, combo_pred.values))

print('5-fold combined accuracy:', np.mean(combo_accs), '+/-', np.std(combo_accs))

5-fold combined accuracy: 0.9402083333333333 +/- 0.005376453291901661


### Fit on full training data, predict on test, write submission

In [4]:
word_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=3, sublinear_tf=True)

X_train, X_test = build_features(train['text'], test['text'],
                                  train['complaint_history'], test['complaint_history'],
                                  word_vec, char_vec)

cat_model = LinearSVC(class_weight='balanced', C=1.0).fit(X_train, train['category'])
urg_model = LinearSVC(class_weight='balanced', C=1.0).fit(X_train, train['urgency'])

cat_pred_tfidf = cat_model.predict(X_test)
urg_pred_tfidf = urg_model.predict(X_test)

submission = pd.DataFrame({
    'id': test['id'],
    'label': [f"{c}|{u}" for c, u in zip(cat_pred_tfidf, urg_pred_tfidf)]
})

sample = pd.read_csv('sample_submission.csv')
assert list(submission.columns) == list(sample.columns)
assert len(submission) == len(sample)
assert set(submission['id']) == set(sample['id'])

submission.to_csv('submission_baseline_tfidf.csv', index=False)
files.download('submission_baseline_tfidf.csv')
submission.head()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,id,label
0,G105529,agriculture_irrigation|high
1,G105472,electricity|routine
2,G103379,roads_transport|high
3,G101632,water_supply|critical
4,G106613,roads_transport|critical


## 3. (Optional, needs GPU) Fine-tune MuRIL for urgency only
Only run this section if `Runtime → Change runtime type → GPU` is enabled.
This only targets the 3-class **urgency** subtask — category is already
solved above, so there's no reason to spend GPU time re-solving it.

MuRIL (`google/muril-base-cased`) is built for Indian languages including
romanized/transliterated text, which matches this dataset
("Sir namma area alli neeru illa"). Try `xlm-roberta-base` as a fallback if
MuRIL underperforms.

**Rule of thumb:** only keep this model if its validation accuracy genuinely
beats the ~0.94 TF-IDF baseline above. A transformer that ties or loses isn't
worth the added complexity you'd have to explain in the interview.

In [5]:
!pip install -q transformers datasets accelerate

import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from datasets import Dataset
from sklearn.metrics import f1_score

print("GPU available:", torch.cuda.is_available())

MODEL_NAME = "google/muril-base-cased"   # fallback: "xlm-roberta-base"
MAX_LEN = 160

urg2id = {'critical': 0, 'high': 1, 'routine': 2}
id2urg = {v: k for k, v in urg2id.items()}
train['urg_id'] = train['urgency'].map(urg2id)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN, padding='max_length')

GPU available: True


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

In [6]:
tr_idx, va_idx = next(
    StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ).split(train, train['urgency'])
)

tr_df = train.iloc[tr_idx].reset_index(drop=True)
va_df = train.iloc[va_idx].reset_index(drop=True)

tr_ds = (
    Dataset.from_pandas(tr_df[['text', 'urg_id']])
    .rename_column('urg_id', 'labels')
)

va_ds = (
    Dataset.from_pandas(va_df[['text', 'urg_id']])
    .rename_column('urg_id', 'labels')
)

tr_ds = tr_ds.map(tokenize, batched=True)
va_ds = va_ds.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro')
    }

args = TrainingArguments(
    output_dir='urgency_model',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=500,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    fp16=torch.cuda.is_available(),
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tr_ds,
    eval_dataset=va_ds,
    compute_metrics=compute_metrics
)

trainer.train()

results = trainer.evaluate()
print(results)

Map:   0%|          | 0/3840 [00:00<?, ? examples/s]

Map:   0%|          | 0/960 [00:00<?, ? examples/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  953MB            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.877106,0.347787,4,1.000000,1.000000


{'eval_loss': 0.34778672456741333, 'eval_accuracy': 1.0, 'eval_macro_f1': 1.0}


## 4. Combine transformer urgency + TF-IDF category → final submission
Only run this if the transformer beat the TF-IDF urgency baseline above.

In [7]:
te_ds = Dataset.from_pandas(test[['text']]).map(tokenize, batched=True)
urg_logits = trainer.predict(te_ds).predictions
urg_pred_transformer = [id2urg[i] for i in np.argmax(urg_logits, axis=1)]

submission_transformer = pd.DataFrame({
    'id': test['id'],
    'label': [f"{c}|{u}" for c, u in zip(cat_pred_tfidf, urg_pred_transformer)]
})
submission_transformer.to_csv('submission_transformer.csv', index=False)
files.download('submission_transformer.csv')
submission_transformer.head()

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,id,label
0,G105529,agriculture_irrigation|high
1,G105472,electricity|routine
2,G103379,roads_transport|high
3,G101632,water_supply|critical
4,G106613,roads_transport|critical
